# Reduced Error Pruning (Validation-Based) - Starter Notebook
This notebook uses a validation set to pick a pruned decision tree with cost-complexity pruning, which is a practical reduced-error pruning style workflow.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

X_train.shape, X_val.shape, X_test.shape

In [ ]:
base_tree = DecisionTreeClassifier(random_state=42)
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

records = []
for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    model.fit(X_train, y_train)
    val_score = accuracy_score(y_val, model.predict(X_val))
    records.append((alpha, val_score, model))

best_alpha, best_val_score, best_model = max(records, key=lambda x: x[1])
print(f"Best ccp_alpha from validation: {best_alpha:.6f}")
print(f"Validation accuracy: {best_val_score:.3f}")

In [ ]:
unpruned = DecisionTreeClassifier(random_state=42)
unpruned.fit(X_train, y_train)

unpruned_test = accuracy_score(y_test, unpruned.predict(X_test))
pruned_test = accuracy_score(y_test, best_model.predict(X_test))

print(f"Unpruned test accuracy: {unpruned_test:.3f}")
print(f"Pruned test accuracy:   {pruned_test:.3f}")

summary_df = pd.DataFrame({
    "model": ["unpruned", "pruned"],
    "test_accuracy": [unpruned_test, pruned_test]
})
summary_df